## Threshold Tuning

Threshold tuning is the process of choosing the best probability cutoff for converting predicted probabilities into class labels.

Most classifiers output probabilities, not directly class labels.

Example

Patient A → 0.92

Patient B → 0.67

Patient C → 0.48

Patient D → 0.25

To convert these into class labels: positive or negative

we choose a threshold. Usually, Threshold = 0.5

If Probability ≥ 0.5 ->predict positive otherwise Negative

## Why do we need Threshold Tuning?

The default threshold of 0.5 is not always the best choice.It is simply a convention.

Different problems have different costs.

Example: Disease Detection

Missing a patient is much worse than sending one healthy patient for another test.

Therefore, we may choose Threshold = 0.30 instead of 0.50

## Goal of Threshold Tuning

Find the threshold that gives the best performance for your objective.

Objectives may include:

1-Maximum F1-score

2-Maximum Recall

3-Maximum Precision

4-Maximum Accuracy

5-Lowest Cost

6-Highest Business Profit

In [1]:
# Model Output Before Threshold

# Suppose a model predicts

# Person	Probability
# A	        0.95
# B	        0.81
# C	        0.67
# D	        0.49
# E	        0.33
# F	        0.12

# Notice These are probabilities, not classes.

# Threshold converts them into classes.

# Example using Threshold = 0.50
# Probability	Prediction
# 0.95	        Positive
# 0.81	        Positive
# 0.67	        Positive
# 0.49	        Negative
# 0.33	        Negative
# 0.12	        Negative

# Example using Threshold = 0.30
# Probability	Prediction
# 0.95	        Positive
# 0.81	        Positive
# 0.67	        Positive
# 0.49	        Positive
# 0.33	        Positive
# 0.12	        Negative

# Notice Lower threshold means More positives predicted.

## What Changes When Threshold Changes?

Changing the threshold does not retrain the model.

Only the final decision changes.

Model probabilities remain exactly the same.

## How Threshold Affects Metrics

***Lower Threshold:***

Recall ↑

Precision ↓

False Positives ↑

False Negatives ↓

***Higher Threshold:***

Precision ↑

Recall ↓

False Positives ↓

False Negatives ↑

Choosing Threshold using F1-score

One common approach

Try many thresholds

0.05

0.10

0.15

...

0.95

Calculate

Precision

Recall

F1

Choose

Maximum F1.

In [8]:
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score,accuracy_score,precision_score,recall_score
## Step 1: create a dataset
X,y=make_classification(
    n_features=20,
    n_samples=10000,
    n_informative=10,
    n_redundant=5,
    random_state=42,
    n_classes=2
)

from sklearn.model_selection import train_test_split
## Step 2: Split into train,test and validation split
X_train,X_tmp,y_train,y_tmp=train_test_split(X,y,test_size=0.4,stratify=y,random_state=42)
X_val,X_test,y_val,y_test=train_test_split(X_tmp,y_tmp,test_size=0.5,stratify=y_tmp,random_state=42)

from sklearn.naive_bayes import GaussianNB
## Step 3: Train Naive Bayes
nb=GaussianNB()
nb.fit(X_train,y_train)

## Step 4: Predict validation probabilites
val_probs=nb.predict_proba(X_val)[:,1]


from sklearn.linear_model import LogisticRegression
## Step 5: Train Platt Scaling
platt=LogisticRegression()
platt.fit(
    val_probs.reshape(-1,1),
    y_val
)


val_probs=platt.predict_proba(nb.predict_proba(X_val)[:,1].reshape(-1,1))[:,1]

best_threshold=0

best_f1=0

for t in np.arange(0.05,1.0,0.01):

    preds=(val_probs > t).astype(int)
    score=f1_score(y_val,preds)

    if score > best_f1:
        best_f1=score
        best_threshold=t
print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

Best Threshold: 0.32000000000000006
Best F1: 0.811787072243346


In [7]:

test_prob=platt.predict_proba(nb.predict_proba(X_test)[:,1].reshape(-1,1))[:,1]

test_pred = (test_prob >= best_threshold).astype(int)

f1_score(y_test,test_pred)

0.8051823416506718

In [ ]:
## Using a function to find optimal thresholds
def best_threshold(y_true,probabilites,metric="f1",start=0.05,end=0.95,step=0.01):
    thresholds=np.arange(start,start+end,step)
    results=[]

    for t in thresholds:
        pred=(probabilites >= t).astype(int)
        accuracy=accuracy_score(y_true,pred)
        precision=precision_score(y_true,pred)
        recall=recall_score(y_true,pred)
        f1=f1_score(y_true,pred)
        results.append({
            "threshold":t,
            "accuracy":accuracy,
            "Precision":precision,
            "recall":recall,
            "f1":f1
        }
        )
    results=pd.DataFrame(results)
    best_row=results.loc[results[metric].idxmax()]
    return best_row,results


best,history=best_threshold(y_val,val_probs,metric='f1')
print(history)
print("best_threshold:",best['threshold'])



    threshold  accuracy  Precision  recall        f1
0        0.05    0.4995     0.4995     1.0  0.666222
1        0.06    0.4995     0.4995     1.0  0.666222
2        0.07    0.4995     0.4995     1.0  0.666222
3        0.08    0.4995     0.4995     1.0  0.666222
4        0.09    0.4995     0.4995     1.0  0.666222
..        ...       ...        ...     ...       ...
90       0.95    0.5005     0.0000     0.0  0.000000
91       0.96    0.5005     0.0000     0.0  0.000000
92       0.97    0.5005     0.0000     0.0  0.000000
93       0.98    0.5005     0.0000     0.0  0.000000
94       0.99    0.5005     0.0000     0.0  0.000000

[95 rows x 5 columns]
best_threshold: 0.32000000000000006


/Users/priyanshugupta/Desktop/scikit learn/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/priyanshugupta/Desktop/scikit learn/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/priyanshugupta/Desktop/scikit learn/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

## using precision_recall curve


In [ ]:
from sklearn.metrics import precision_recall_curve
def best_threshold(y_true,probabilites,start=0.05,end=0.95,step=0.01):

    prec,recall,thresholds=precision_recall_curve(y_true,probabilites)
    f1= 2 * prec[:-1]* recall[:-1]/ (prec[:-1] + recall[:-1] + 1e-9)
    best_idx=np.argmax(f1)
    threshold=thresholds[best_idx]
    return threshold

t=best_threshold(y_val,val_probs)
t


np.float64(0.32522910870941674)

In [ ]:
# 1. precision_recall_curve()
# prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
# print(len(prec))
# print(len(rec))
# print(len(thresholds))
# Output is always:
# len(prec) = N + 1
# len(rec) = N + 1
# len(thresholds) = N
# Notice:
# precision has one extra element.
# recall has one extra element.
# thresholds is one element shorter.
# That's why we do:
# f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
# Now
# len(f1) == len(thresholds)
# so
# best_t = thresholds[best_idx]
# works correctly.
# Why does precision_recall_curve() add an extra element?
# The last point corresponds to a classifier that predicts every sample as positive (recall = 1.0). There is no threshold associated with this endpoint, so scikit-learn appends an extra precision/recall value but does not append another threshold.
# For example:
# precision
# [1.00, 0.90, 0.82, 0.75]
# recall
# [0.20, 0.55, 0.80, 1.00]
# thresholds
# [0.82, 0.54, 0.31]

# Notice:
# precision  -> 4 values
# recall     -> 4 values
# thresholds -> 3 values

## Using roc_curve

In [12]:
from sklearn.metrics import roc_curve
def best_threshold(y_true,probabilites,start=0.05,end=0.95,step=0.01):
    fpr,tpr,thresholds=roc_curve(y_true,probabilites)
    j_scores=tpr-fpr
    best_idx=np.argmax(j_scores)
    threshold=thresholds[best_idx]
    return threshold

t=best_threshold(y_val,val_probs)
print(t)

0.40199429581790447
